In [1]:
import earthaccess
from dist_s1_enumerator.mgrs_burst_data import get_mgrs_table
import pandas as pd
from tqdm import tqdm
from pathlib import Path

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_mgrs = get_mgrs_table()

In [3]:
earthaccess.login()

# Testing

In [4]:
def get_opera_id(query_item) -> str:
    return dict(query_item.__dict__['render_dict'])['meta']['native-id']

def july_query_dist_hls_cmr(mgrs_tile_id: str, start_time='2025-07-01', stop_time='2025-08-01'):
    collection_short_name = 'OPERA_L3_DIST-ALERT-HLS_V1'
    mgrs_bounds = tuple(df_mgrs[df_mgrs.mgrs_tile_id == mgrs_tile_id].total_bounds)
    mgrs_bounds = tuple(float(x) for x in mgrs_bounds)
    datasets_found = earthaccess.search_data(
        short_name=collection_short_name,
        temporal=(start_time, stop_time),
        cloud_hosted=True,
        bounding_box=mgrs_bounds
    )
    datasets_found = [d for d in datasets_found if mgrs_tile_id in get_opera_id(d)]
    return datasets_found

In [9]:
q = july_query_dist_hls_cmr('18SUJ')


In [7]:
def get_processing_time(query_item: earthaccess.DataGranule) -> pd.Timestamp:
    data = query_item.__dict__['render_dict']
    ts = pd.Timestamp(data['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'])
    return ts

In [11]:
#dict(qs_ordered[0].__dict__['render_dict'])

In [10]:
qs_ordered = sorted(q, key=get_processing_time)

In [12]:
# earthaccess.download(q[0], '.')

# Automate

In [13]:
df_data = pd.read_csv('amy_val_rest_of_informal_sites_10_2025.csv')
names = df_data.name.unique().tolist()
names

['fire__29TNF',
 'logging__10TES',
 'logging__33NZD',
 'logging__33VWJ',
 'mining__19LCF',
 'mining__35NRC',
 'mining__52NCF',
 'road_expansion__50SLD',
 'shifting_cultivation__34MHE',
 'tornado__16RBV',
 'tornado__16SDB']

In [14]:
def get_most_recent_july_date(mgrs_tile_id: str, job_name: str) -> str:
    datasets = july_query_dist_hls_cmr(mgrs_tile_id)
    if datasets:
        datasets_ordered = sorted(datasets, key=get_processing_time)
        opera_id = dict(datasets_ordered[-1].__dict__['render_dict'])['meta']['native-id']
        out_dir = Path(f'dist_hls/{job_name}/{opera_id}')
        out_dir.mkdir(exist_ok=True, parents=True)
        r = earthaccess.download(datasets_ordered[-1], out_dir)
        return r
    else:
        return ''

In [15]:
mgrs_tile_ids = [name.split('__')[-1] for name in names]
mgrs_tile_ids[:1]

['29TNF']

In [16]:
_ = [get_most_recent_july_date(m_id, name) for m_id, name in zip(mgrs_tile_ids, tqdm(names[:]))]

  0%|                           | 0/11 [00:00<?, ?it/s]
QUEUEING TASKS | : 100%|█| 19/19 [00:00<00:00, 4044.04i

PROCESSING TASKS | :   0%|      | 0/19 [00:00<?, ?it/s]
PROCESSING TASKS | :   5%| | 1/19 [00:02<00:39,  2.22s/
PROCESSING TASKS | :  16%|▏| 3/19 [00:02<00:09,  1.61it
PROCESSING TASKS | :  26%|▎| 5/19 [00:02<00:04,  2.87it
PROCESSING TASKS | :  37%|▎| 7/19 [00:02<00:02,  4.32it
PROCESSING TASKS | :  47%|▍| 9/19 [00:02<00:02,  4.85it
PROCESSING TASKS | :  53%|▌| 10/19 [00:03<00:01,  5.35i
PROCESSING TASKS | :  58%|▌| 11/19 [00:03<00:01,  5.89i
PROCESSING TASKS | :  63%|▋| 12/19 [00:03<00:01,  5.40i
PROCESSING TASKS | :  68%|▋| 13/19 [00:03<00:01,  5.73i
PROCESSING TASKS | :  79%|▊| 15/19 [00:03<00:00,  7.52i
PROCESSING TASKS | :  84%|▊| 16/19 [00:03<00:00,  7.45i
PROCESSING TASKS | : 100%|█| 19/19 [00:04<00:00,  4.42i

COLLECTING RESULTS | : 100%|█| 19/19 [00:00<00:00, 3348
  9%|█▋                 | 1/11 [00:06<01:04,  6.48s/it]
QUEUEING TASKS | : 100%|█| 19/19 [00:00<00:00,